### Phase 1: Environment Setup and Data Ingestion
The objective of this phase is to establish a reproducible training environment and ingest the raw datasets. 

* **Reproducibility**: We set a global random seed to ensure that our train/test splits and model initializations yield consistent results across different runs.
* **Data Loading**: We utilize `pandas` to load the raw `ratings.csv` and `movies.csv` files from our local data directory. This ingestion step acts as the foundation for all subsequent feature engineering and modeling tasks.

In [13]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
import joblib
import os


In [2]:

# Set a random seed for reproducibility
np.random.seed(42)

print("Libraries imported and seed set.")

# Define paths to our data
# We assume the structure: project_root/data/raw/
DATA_PATH = '../data/'

# Load the datasets
ratings = pd.read_csv(os.path.join(DATA_PATH, 'ratings.csv'))
movies = pd.read_csv(os.path.join(DATA_PATH, 'movies.csv'))

# Display the first few rows to verify successful loading
print("Data loaded successfully.")
display(ratings.head())
display(movies.head())

Libraries imported and seed set.
Data loaded successfully.


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### Phase 2: Data Cleaning and Merging
The objective of this phase is to prepare our datasets for modeling by handling missing values, cleaning metadata, and creating a unified dataframe.

* **Data Integrity**: We ensure that all ratings are associated with existing movies by merging on `movieId`.
* **Preprocessing**: We handle any potential `NaN` values and structure the data so it is ready for matrix factorization or other collaborative filtering algorithms.

In [4]:
# Merge ratings and movies dataframes
df = pd.merge(ratings, movies, on='movieId')

# Check for missing values
print("Missing values in the merged dataset:")
print(df.isnull().sum())

# Drop rows with missing values if any
df = df.dropna()

# Display the merged dataframe structure
print("Merged dataset shape:", df.shape)
display(df.head())

Missing values in the merged dataset:
userId       0
movieId      0
rating       0
timestamp    0
title        0
genres       0
dtype: int64
Merged dataset shape: (100836, 6)


,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


### Phase 3: Train-Test Splitting
The objective of this phase is to partition our interaction data into training and testing sets. This allows us to train the model on one subset of data and evaluate its predictive performance on unseen data, which is vital for preventing overfitting.

* **Split Strategy**: We will perform a chronological or random split to ensure the model generalizes well to new, unseen user-item interactions.
* **Validation**: By holding out a portion of the data, we can accurately measure the model's accuracy (e.g., using RMSE) before deploying it in a production environment.

In [6]:
# Define features and target (user-item-rating triplets)
X = df[['userId', 'movieId']]
y = df['rating']

# Perform a 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (80668, 2)
Testing set shape: (20168, 2)


### Phase 4: Model Selection Strategy
The objective of this phase is to evaluate which algorithm best suits our dataset. We choose between SVD and KNN based on how they handle the sparsity and structure of user-item interactions.

* **SVD (Singular Value Decomposition)**: A matrix factorization technique that condenses user-item matrices into "latent factors" (hidden preferences). It is highly efficient for large datasets and excels at capturing subtle patterns in user behavior, making it the industry standard for collaborative filtering.
* **KNN (K-Nearest Neighbors)**: A memory-based approach that identifies "neighbors"—similar users or similar movies—based on shared ratings. It is intuitive and easy to explain, but it can become computationally expensive and slow as the dataset grows.

### Phase 5 (Alternative): Implementing SVD with Scikit-Learn
While the `Surprise` library is purpose-built for recommendation systems, you can certainly implement SVD using `scikit-learn` if you prefer to keep your dependencies unified or if you want more control over the matrix factorization process. In `scikit-learn`, this is typically handled via `TruncatedSVD`.

* **Matrix Representation**: To use `sklearn`, you must first convert your dataframe into a sparse user-item matrix (Utility Matrix), where rows represent users, columns represent movies, and values represent ratings.
* **Dimensionality Reduction**: `TruncatedSVD` works by decomposing this sparse matrix into latent factors.
* **Integration**: Using `sklearn` allows you to easily pipe this step into other machine learning workflows (like `Pipeline` or `GridSearchCV`) that you might already be using in your project.

In [7]:
#!pip install scikit-surprise

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 11.7 MB/s  0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [10]:
# 1. Pivot the data to create a User-Item Matrix
user_movie_matrix = df.pivot(index='userId', columns='movieId', values='rating').fillna(0)

# 2. Convert to a sparse matrix for memory efficiency
sparse_matrix = csr_matrix(user_movie_matrix.values)

# 3. Apply TruncatedSVD
# n_components is the number of latent factors
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_factorized = svd.fit_transform(sparse_matrix)

print("Matrix factorized shape:", matrix_factorized.shape)

Matrix factorized shape: (610, 50)


### Phase 6: Predicting Ratings via Reconstructed Matrix
The objective of this phase is to transform our reduced latent factors back into a "predicted rating" matrix. Because `TruncatedSVD` compresses the data, the resulting dot product provides an approximation of the original user-item matrix, filling in the "missing" values (the movies the user has not yet rated).

* **Reconstruction**: By calculating the dot product of our components (`UΣ` and `V^T`), we generate estimated ratings for all user-movie pairs.
* **Inference**: Once we have this reconstructed matrix, we can filter for specific users to identify the highest-rated movies they haven't seen yet.

In [12]:
# 1. Reconstruct the ratings matrix
# This projects the reduced features back into the original movie space
predicted_ratings = svd.inverse_transform(matrix_factorized)

# 2. Convert to DataFrame for easier lookups
predicted_df = pd.DataFrame(
    predicted_ratings, 
    columns=user_movie_matrix.columns, 
    index=user_movie_matrix.index
)

# 3. Define a function to get recommendations for a user
def get_recommendations(user_id, predicted_df, n=10):
    # Get the user's predicted ratings
    user_predictions = predicted_df.loc[user_id]
    
    # Sort the movies by predicted rating in descending order
    recommendations = user_predictions.sort_values(ascending=False)
    
    return recommendations.head(n)

# Example: Get recommendations for user 1
print("Top 10 recommendations for user 1:")
print(get_recommendations(1, predicted_df))

Top 10 recommendations for user 1:
movieId
260     6.923456
1196    6.644699
1198    6.611688
1210    6.124899
1291    5.987938
608     5.514544
47      5.203002
296     5.122181
2115    5.055617
2858    5.043141
Name: 1, dtype: float64


### Analysis: Understanding the SVD Prediction Scores
The numbers you see represent the **predicted preference score** (or "estimated rating") that the SVD algorithm has assigned to each movie for user 1.

* **What they represent:** These are not raw star ratings (e.g., 1–5). Instead, they are the result of the matrix reconstruction process. Because `TruncatedSVD` is a mathematical approximation, it can generate scores that fall outside your original 1–5 range.
* **Interpretation:** Think of these as a **ranking index**. A higher number indicates that the algorithm believes the user will have a stronger preference for that movie relative to others. 
* **Why they aren't 1–5:** When `TruncatedSVD` projects your data into latent space and then reconstructs it, it does not have a "clamp" to keep values between 1 and 5. This is normal behavior for this algorithm.

**Important Note for the Frontend:**
Since these numbers (like 6.92) are not standard 5-star ratings, you should **normalize** them before displaying them to a user in your UI. You can use Min-Max scaling to map these back to a 1–5 scale, or simply use them to rank the movies and hide the actual score from the user.

### Phase 7: Model Serialization
The objective of this phase is to persist our trained SVD model and the resulting recommendation dataframe to disk. By saving these as binary files, we can load them directly into your production backend (e.g., a FastAPI or Next.js API route) without needing to retrain the model every time the server starts.

* **Artifacts**: We save the `svd` object (the model logic) and the `predicted_df` (the pre-computed scores).
* **Storage**: We use `joblib` for efficient serialization of large numpy-based objects, which is standard practice in Python data science workflows.

In [14]:

# 1. Create a models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# 2. Serialize the model and the predictions
# Saving the svd object allows you to project new data later
joblib.dump(svd, '../models/svd_model.pkl')

# Saving the predicted_df allows for instant lookups in your API
joblib.dump(predicted_df, '../models/predicted_df.pkl')

print("Model and prediction dataframe saved to ../models/")

Model and prediction dataframe saved to ../models/
